In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import requests
import random
import itertools
import numpy as np
from joblib import Parallel, delayed
import urllib3
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "http://localhost:5222"
BATTLE_ENDPOINT = f"{BASE_URL}/BattleCalculator/calculate-specific-units-team"

UNIT_TYPES = {
    0: "Light",
    1: "Heavy",
    2: "Fast",
    3: "ShortRange",
    4: "LongRange"
}

In [3]:
def generate_random_team(team_size=8):
    return [random.choice(list(UNIT_TYPES.keys())) for _ in range(team_size)]


def run_simulation(team_a, team_b):
    payload = {
        "TeamA": {"Name": "TeamA", "Units": team_a},
        "TeamB": {"Name": "TeamB", "Units": team_b}
    }
    try:
        response = requests.post(BATTLE_ENDPOINT, json=payload, verify=False)
        response.raise_for_status()
        return response.json()
    except:
        return None

In [4]:
def extract_metrics(res):
    def team_metrics(team):
        units = team.get("Units", [])

        damage_dealt = sum(u.get("DamageDealt", 0) for u in units)
        damage_taken = sum(u.get("DamageTaken", 0) for u in units)
        alive = sum(1 for u in units if u.get("Health", 0) > 0)

        return {
            "damage_dealt": damage_dealt,
            "damage_taken": damage_taken,
            "alive": alive
        }

    return {
        "A": team_metrics(res["TeamA"]),
        "B": team_metrics(res["TeamB"])
    }

In [5]:
def compute_score(metrics, alpha=0.5, beta=50):
    return (
        metrics["damage_dealt"]
        - alpha * metrics["damage_taken"]
        + beta * metrics["alive"]
    )

In [6]:
def evaluate_match(res):
    m = extract_metrics(res)

    score_a = compute_score(m["A"])
    score_b = compute_score(m["B"])

    return {
        "score_a": score_a,
        "score_b": score_b,
        "delta": score_a - score_b,
        "team_a": res["TeamA"]["Units"],
        "team_b": res["TeamB"]["Units"]
    }

In [7]:
def simulate_one(_):
    team_a = generate_random_team()
    team_b = generate_random_team()

    res = run_simulation(team_a, team_b)
    if res is None:
        return None

    return evaluate_match(res)


def run_experiment(n=1000, jobs=4):
    results = Parallel(n_jobs=jobs)(
        delayed(simulate_one)(i) for i in range(n)
    )

    results = [r for r in results if r is not None]
    return pd.DataFrame(results)

In [8]:
def compute_unit_efficiency(df):
    stats = {k: {"score": 0, "games": 0} for k in UNIT_TYPES}

    for _, row in df.iterrows():
        for u in row["team_a"]:
            stats[u]["score"] += row["score_a"]
            stats[u]["games"] += 1

        for u in row["team_b"]:
            stats[u]["score"] += row["score_b"]
            stats[u]["games"] += 1

    data = []
    for k, v in stats.items():
        avg = v["score"] / v["games"] if v["games"] else 0
        data.append({"unit": UNIT_TYPES[k], "efficiency": avg})

    return pd.DataFrame(data)

In [9]:
def compute_synergy(df):
    synergy = {}

    for _, row in df.iterrows():
        team = row["team_a"]
        score = row["score_a"]

        for pair in itertools.combinations(sorted(team), 2):
            if pair not in synergy:
                synergy[pair] = {"score": 0, "games": 0}

            synergy[pair]["score"] += score
            synergy[pair]["games"] += 1

    data = []
    for pair, v in synergy.items():
        avg = v["score"] / v["games"]
        data.append({
            "pair": f"{UNIT_TYPES[pair[0]]}+{UNIT_TYPES[pair[1]]}",
            "score": avg
        })

    return pd.DataFrame(data).sort_values(by="score", ascending=False)

In [10]:
def extract_combat_points(res):
    data = []

    for team_key in ["TeamA", "TeamB"]:
        for u in res[team_key]["Units"]:
            data.append({
                "damage": u.get("DamageDealt", 0),
                "taken": u.get("DamageTaken", 0),
                "alive": u.get("Health", 0) > 0
            })

    return data


def build_scatter_dataset(n=300):
    rows = []

    for _ in range(n):
        team_a = generate_random_team()
        team_b = generate_random_team()
        res = run_simulation(team_a, team_b)

        if res:
            rows.extend(extract_combat_points(res))

    return pd.DataFrame(rows)

In [11]:
def plot_results(df_units, scatter_df):
    # Efficiency
    plt.figure()
    plt.bar(df_units["unit"], df_units["efficiency"])
    plt.title("Unit Efficiency")
    plt.xticks(rotation=30)
    plt.show()

    # Score distribution
    plt.figure()
    plt.hist(df["delta"], bins=30)
    plt.title("Score Difference Distribution")
    plt.show()

    # Survivability vs Damage
    plt.figure()
    plt.scatter(scatter_df["damage"], scatter_df["taken"])
    plt.title("Damage vs Damage Taken")
    plt.xlabel("Damage Dealt")
    plt.ylabel("Damage Taken")
    plt.show()

In [ ]:
# Run experiment
df = run_experiment(n=1000)
print(df.keys)


In [ ]:

# Compute metrics
unit_df = compute_unit_efficiency(df)
synergy_df = compute_synergy(df)

# Extra dataset
scatter_df = build_scatter_dataset(300)

# Show results
print(unit_df.sort_values(by="efficiency", ascending=False))
print("\nTop Synergies:")
print(synergy_df.head(10))

# Plot
plot_results(unit_df, scatter_df)